# Amazon Bedrock Agents

This notebook demonstrates how to create an Amazon OpenSearch Serverless collection, a Bedrock Knowledge Base, and a Bedrock Agent using the AWS SDK for Python (Boto3).

## Step 1: Install dependencies and initialize clients

In [1]:
# Install required packages: Installs boto3, the AWS SDK for Python, 
# which is needed to interact with AWS services programmatically through API calls.

!pip install boto3

In [2]:
# Import libraries: Imports necessary Python modules for AWS interactions, 
# timing operations for polling, and JSON handling for policy definitions.

import sys
import os

# Silence sagemaker JumpStart stderr noise during import
stderr = sys.stderr
sys.stderr = open(os.devnull, 'w')

import boto3
import time
import json
from sagemaker import get_execution_role

sys.stderr = stderr

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


<i aria-hidden="true" class="fas fa-exclamation-circle" style="color:#7C5AED"></i> **Caution:** Before running the next code cell, replace the following placeholder variables with their actual values that you copied earlier. Those values are provided to the left of the main lab instructions.

- BedrockKnowledgeBaseRoleArn
- S3BucketBKB
- BedrockAgentLambdaCreateITSM
- BedrockAgentLambdaLookupITSM
- BedrockAgentRoleArn

In [3]:
# Initialize clients: Sets up the OpenSearchServerless client and defines 
# key variables including role ARNs and S3 bucket that will be used throughout the notebook. 
# Gets the current region and execution role to ensure proper AWS resource access and permissions.

client = boto3.client('opensearchserverless')

# Update with outputs for lab in CloudFormation
bedrock_bkb_role_arn = 'arn:aws:iam::242734630127:role/BedrockKnowledgeBaseRole'
s3_bucket = 'labstack-a1c260ab-3223-4e5d-bed6-a3637-s3bucketbkb-tzxafgf0vbhp'
lambda_arn_create = 'arn:aws:lambda:eu-west-2:242734630127:function:SkillBuilder-BedrockAgent-CreateITSM-Function'
lambda_arn_lookup = 'arn:aws:lambda:eu-west-2:242734630127:function:SkillBuilder-BedrockAgent-LookupITSM-Function'
bedrock_agent_role_arn = 'arn:aws:iam::242734630127:role/BedrockAgentRole'

# Get current region 
curr_session = boto3.session.Session()
curr_region = curr_session.region_name

# Get current execution role
exec_role = get_execution_role()

## Step 2: Create a network policy for the collection

In [4]:
# Define collection name: Names the OpenSearch Serverless collection 
# that will be created and used throughout the lab.

collection_name = 'skillbuilder'

# Create network policy: Creates a security policy that allows public access 
# to the collection and dashboard, which is necessary for Bedrock to access 
# the vector store. This enables the Knowledge Base to query the collection.

network_policy_json = json.dumps([
    {
        "Rules": [
            {
                "ResourceType": "dashboard",
                "Resource": [f"collection/{collection_name}"],
            },
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"],
            }
        ],
        "AllowFromPublic": True
    }
])

try:
    response = client.create_security_policy(
        name=f"{collection_name}-network-policy",
        type='network',
        description='Network policy for OpenSearch Serverless collection',
        policy=network_policy_json
    )
    print(f"Network policy created: {response['securityPolicyDetail']['name']}")
except client.exceptions.ConflictException:
    print(f"Network policy '{collection_name}-network-policy' already exists, skipping.")

Network policy created: skillbuilder-network-policy


## Step 3: Create a data access policy for the collection

In [5]:
# Create data access policy: Defines permissions for who can access the collection data. 
# Grants full permissions to both the Bedrock Knowledge Base role and the notebook's 
# execution role to perform operations on the collection and its indexes. 
# This ensures proper authorization for data operations.

data_policy_json = json.dumps([
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"],
                "Permission": [
                    "aoss:*"
                ]
            },
            {
                "ResourceType": "index",
                "Resource": [f"index/{collection_name}/*"],
                "Permission": [
                    "aoss:*"
                ]
            }
        ],
        "Principal": [bedrock_bkb_role_arn, exec_role]
    }
])

try:
    response = client.create_access_policy(
        name=f"{collection_name}-data-policy",
        type='data',
        description='Data policy for OpenSearch Serverless collection',
        policy=data_policy_json
    )
    print(f"Data policy created: {response['accessPolicyDetail']['name']}")
except client.exceptions.ConflictException:
    print(f"Data policy '{collection_name}-data-policy' already exists, skipping.")

Data policy created: skillbuilder-data-policy


## Step 4: Create an encryption policy

In [6]:
# Create an encryption policy: Sets up encryption for the OpenSearch collection 
# using AWS-owned keys, ensuring data is encrypted at rest as a security best practice. 
# This protects the vector data stored in the collection.

encryption_policy_json = json.dumps(
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"]
            }
        ],
        "AWSOwnedKey": True
    }
)

try:
    response = client.create_security_policy(
        name=f"{collection_name}-encryption-policy",
        type="encryption",
        description="Encryption policy for OpenSearch Serverless collection",
        policy=encryption_policy_json
    )
    print(f"Encryption policy created: {response['securityPolicyDetail']['name']}")
except client.exceptions.ConflictException:
    print(f"Encryption policy '{collection_name}-encryption-policy' already exists, skipping.")

Encryption policy created: skillbuilder-encryption-policy


## Step 5: Create the OpenSearch Serverless collection

In [7]:
# Create the collection: Initializes a VECTORSEARCH type OpenSearch Serverless 
# collection that will store vector embeddings for the Knowledge Base. 
# This specialized collection type is optimized for vector similarity searches.

try:
    response = client.create_collection(
        name=collection_name,
        type='VECTORSEARCH',
        description='OpenSearch Serverless collection for a Bedrock Knowledge Base'
    )
    collection_id = response['createCollectionDetail']['id']
    print(f"Collection creation initiated. Collection ID: {collection_id}")
except client.exceptions.ConflictException:
    response = client.batch_get_collection(names=[collection_name])
    collection_id = response['collectionDetails'][0]['id']
    print(f"Collection '{collection_name}' already exists. Collection ID: {collection_id}")

Collection creation initiated. Collection ID: 3oyk67yzw59kwm0xwbke


In [8]:
# Wait for collection: Implements a polling mechanism to check the collection's 
# status until it becomes ACTIVE, as collection creation is an asynchronous operation 
# that can take several minutes to complete.

print("Waiting for collection to become active...")
status = None
while True:
    response = client.batch_get_collection(ids=[collection_id])
    status = response['collectionDetails'][0]['status']
    print(f"Current status: {status}")
    
    if status == 'ACTIVE':
        break
    elif status in ['FAILED', 'DELETED']:
        print(f"Collection creation failed with status: {status}")
        break
        
    time.sleep(15)

Waiting for collection to become active...
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: ACTIVE


## Step 6: Get collection endpoint

In [9]:
# Get collection endpoint: Retrieves and displays the endpoints for the 
# collection and dashboard once the collection is active, which will be needed 
# to configure the Knowledge Base connection to OpenSearch Serverless.

if status == 'ACTIVE':
    response = client.batch_get_collection(ids=[collection_id])
    dashboard_endpoint = response['collectionDetails'][0]['dashboardEndpoint']
    collection_endpoint = response['collectionDetails'][0]['collectionEndpoint']
    
    print(f"\nCollection successfully created!")
    print(f"Collection Name: {collection_name}")
    print(f"Collection ID: {collection_id}")
    print(f"Dashboard Endpoint: {dashboard_endpoint}")
    print(f"Collection Endpoint: {collection_endpoint}")
    print(f"Bedrock Knowledge Base Role ARN: {bedrock_bkb_role_arn}")


Collection successfully created!
Collection Name: skillbuilder
Collection ID: 3oyk67yzw59kwm0xwbke
Dashboard Endpoint: https://3oyk67yzw59kwm0xwbke.eu-west-2.aoss.amazonaws.com/_dashboards
Collection Endpoint: https://3oyk67yzw59kwm0xwbke.eu-west-2.aoss.amazonaws.com
Bedrock Knowledge Base Role ARN: arn:aws:iam::242734630127:role/BedrockKnowledgeBaseRole


## Step 7: Create the vector index

In [10]:
# Install required packages for OpenSearch : Installs the Python requests 
# and requests-aws4auth libraries needed to make authenticated API calls 
# to OpenSearch Serverless.

!pip install requests requests-aws4auth

In [11]:
import requests
from requests_aws4auth import AWS4Auth

# Create AWS authentication for OpenSearch : Sets up AWS Signature Version 4 
# authentication (AWS4Auth) which is required to securely access the OpenSearch APIs. 
# This uses your current AWS credentials to sign requests to the OpenSearch Serverless endpoints.

credentials = boto3.Session().get_credentials()
region = boto3.Session().region_name
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    'aoss',
    session_token=credentials.token
)

### Define the index mapping with vector field for Amazon Titan V2 Embeddings

In [12]:
# Define the index mapping with vector field: Creates a specialized index configuration 
# optimized for vector embeddings from Amazon Titan V2. The mapping includes:

# A knn_vector field with 1024 dimensions (matching Titan V2's embedding size)

# FAISS as the vector search engine with HNSW algorithm for efficient similarity search

# Special fields for storing document text chunks and metadata that Bedrock Knowledge Base will use

index_mapping = {
  "settings": {
    "index": {
      "knn": "true"
    }
  },
  "mappings": {
    "dynamic_templates": [
      {
        "strings": {
          "match_mapping_type": "string",
          "mapping": {
            "fields": {
              "keyword": {
                "ignore_above": 2147483647,
                "type": "keyword"
              }
            },
            "type": "text"
          }
        }
      }
    ],
    "properties": {
      "bedrock-knowledge-base-default-vector": {
        "type": "knn_vector",
        "dimension": 1024,
        "method": {
          "engine": "faiss",
          "space_type": "l2",
          "name": "hnsw",
          "parameters": {}
        }
      },
      "AMAZON_BEDROCK_METADATA": {
        "type": "text",
        "index": "false"
      },
      "AMAZON_BEDROCK_TEXT_CHUNK": {
        "type": "text",
        "index": "true"
      }
    }
  }
}

In [13]:
# Create the index
index_name = f"{collection_name}-index"

# Print the endpoint to debug
print(f"Collection endpoint: {collection_endpoint}")

# Fix URL construction
if collection_endpoint.startswith('https://'):
    url = f"{collection_endpoint}/{index_name}"
else:
    url = f"https://{collection_endpoint}/{index_name}"

print(f"Using URL: {url}")

# Create the index with a Put request
response = requests.put(url, auth=awsauth, json=index_mapping, headers={"Content-Type": "application/json"})
print(f"Index creation response: {response.status_code}")
print(response.text)

Collection endpoint: https://3oyk67yzw59kwm0xwbke.eu-west-2.aoss.amazonaws.com
Using URL: https://3oyk67yzw59kwm0xwbke.eu-west-2.aoss.amazonaws.com/skillbuilder-index
Index creation response: 200
{"acknowledged":true,"shards_acknowledged":true,"index":"skillbuilder-index"}


## Step 8: Create a Bedrock Knowledge Base

In [14]:
# Initialize Bedrock client - Creates a boto3 client specifically for interacting 
# with the Bedrock Agent service.

bedrock_client = boto3.client('bedrock-agent')

In [15]:
# Create a Bedrock Knowledge Base: Sets up a Knowledge Base that uses the OpenSearch Serverless 
# collection as its vector store and configures it to use Amazon Titan Embeddings V2 model 
# for converting text to vectors.

response = bedrock_client.create_knowledge_base(
    name=f"{collection_name}-kb",
    description="Knowledge base for SkillBuilder lab",
    roleArn=bedrock_bkb_role_arn,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": f"arn:aws:bedrock:{curr_region}::foundation-model/amazon.titan-embed-text-v2:0"
        }
    },
    storageConfiguration={
        "type": "OPENSEARCH_SERVERLESS",
        "opensearchServerlessConfiguration": {
            "collectionArn": f"arn:aws:aoss:{curr_region}:{boto3.client('sts').get_caller_identity()['Account']}:collection/{collection_id}",
            "vectorIndexName": f"{collection_name}-index",
            "fieldMapping": {
                "vectorField": "bedrock-knowledge-base-default-vector",
                "textField": "AMAZON_BEDROCK_TEXT_CHUNK",
                "metadataField": "AMAZON_BEDROCK_METADATA"
            }
        }
    }
)

knowledge_base_id = response['knowledgeBase']['knowledgeBaseId']
print(f"Knowledge Base creation initiated. Knowledge Base ID: {knowledge_base_id}")

Knowledge Base creation initiated. Knowledge Base ID: 2SZKIHRUWI


<i aria-hidden="true" class="fas fa-exclamation-circle" style="color:#7C5AED"></i> **Caution:** If you see the following error, please re-run the code cell:

*ValidationException: An error occurred (ValidationException) when calling the CreateKnowledgeBase operation: The 
knowledge base storage configuration provided is invalid... Dependency error document status code: 404, error 
message: no such index [skillbuilder-index]*

In [16]:
# Wait for Knowledge Base activation: Implements a polling mechanism to check 
# the Knowledge Base status until it becomes active, as creation is an asynchronous operation.

print("Waiting for knowledge base to become active...")
kb_status = None
while True:
    response = bedrock_client.get_knowledge_base(knowledgeBaseId=knowledge_base_id)
    kb_status = response['knowledgeBase']['status']
    print(f"Current status: {kb_status}")
    
    if kb_status == 'ACTIVE':
        break
    elif kb_status in ['FAILED', 'DELETED']:
        print(f"Knowledge Base creation failed with status: {kb_status}")
        break
        
    time.sleep(30)

Waiting for knowledge base to become active...
Current status: ACTIVE


## Step 9: Create a Data Source and Start Ingestion

In [17]:
# Upload PDF file to S3: Uploads the Employee IT handbook PDF to the S3 bucket that will serve as 
# the data source for the Knowledge Base.

s3_client = boto3.client('s3')
pdf_file_path = 'Fictitious-Company-Employee-IT-Handbook.pdf'
s3_key = 'Fictitious-Company-Employee-IT-Handbook.pdf'

print(f"Uploading {pdf_file_path} to s3://{s3_bucket}/{s3_key}")
s3_client.upload_file(pdf_file_path, s3_bucket, s3_key)
print("Upload complete!")

Uploading Fictitious-Company-Employee-IT-Handbook.pdf to s3://labstack-a1c260ab-3223-4e5d-bed6-a3637-s3bucketbkb-tzxafgf0vbhp/Fictitious-Company-Employee-IT-Handbook.pdf
Upload complete!


In [18]:
# Create a data source: Configures a data source that connects the Knowledge Base 
# to the S3 bucket containing the documents.

response = bedrock_client.create_data_source(
    knowledgeBaseId=knowledge_base_id,
    name=f"{collection_name}-data-source",
    description="S3 data source for the knowledge base",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{s3_bucket}"
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {
                "maxTokens": 300,
                "overlapPercentage": 20
            }
        }
    }
)

data_source_id = response['dataSource']['dataSourceId']
print(f"Data source created. Data Source ID: {data_source_id}")

Data source created. Data Source ID: WINXHK39YQ


In [19]:
# Wait for the data source to become active
print("Waiting for data source to become active...")
ds_status = None
while True:
    response = bedrock_client.get_data_source(knowledgeBaseId=knowledge_base_id, dataSourceId=data_source_id)
    ds_status = response['dataSource']['status']
    print(f"Current status: {ds_status}")
    
    if ds_status == 'AVAILABLE':
        print(f"Current status: {ds_status}")
        break
    elif ds_status in ['DELETING', 'DELETE_UNSUCCESSFUL']:
        print(f"Data source creation failed with status: {ds_status}")
        break
        
    time.sleep(30)

Waiting for data source to become active...
Current status: AVAILABLE
Current status: AVAILABLE


In [20]:
# Start ingestion job: Initiates the process of extracting text from documents, chunking it, 
# creating embeddings, and storing them in the vector index.

response = bedrock_client.start_ingestion_job(
    knowledgeBaseId=knowledge_base_id,
    dataSourceId=data_source_id
)

ingestion_job_id = response['ingestionJob']['ingestionJobId']
print(f"Ingestion job started. Ingestion Job ID: {ingestion_job_id}")

Ingestion job started. Ingestion Job ID: VOAE1HB5HB


In [21]:
# Monitor ingestion job: Tracks the progress of document processing, showing statistics 
# on successful and failed documents.

print("Monitoring ingestion job status...")
job_status = None
while True:
    response = bedrock_client.get_ingestion_job(
        knowledgeBaseId=knowledge_base_id,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id
    )
    job_status = response['ingestionJob']['status']
    statistics = response['ingestionJob'].get('statistics', {})
    
    print(f"Current status: {job_status}")
    if statistics:
        print(f"Documents processed: {statistics.get('numberOfDocumentsProcessed', 0)}")
        print(f"Documents failed: {statistics.get('numberOfDocumentsFailed', 0)}")
    
    if job_status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break
        
    time.sleep(15)

Monitoring ingestion job status...
Current status: COMPLETE
Documents processed: 0
Documents failed: 0


## Step 10: Create a Bedrock Agent

In [22]:
# Create a Bedrock Agent: Sets up an agent using the Nova Lite foundation model with 
# instructions to answer questions about the IT handbook and manage ITSM tickets.

response = bedrock_client.create_agent(
    agentName=f"{collection_name}-agent",
    description="Agent that uses knowledge base and ITSM ticket creation API",
    instruction="You are a helpful assistant that can answer questions about the IT Employee handbook using the knowledge base. You can also create new ITSM tickets using the /create API endpoint or look up ITSM ticket using the /lookup API. When an action group creates a ticket, pass the ticket number you received back to the user. When a user asks you to look up the status, description, urgency, impact, or created date of a ticket number, use the /lookup API. Do not make up any details for the ticket, the details must come from the lookup tool.  Provide all details the user asks for from the lookup tool response.",
    foundationModel="amazon.nova-lite-v1:0",
    idleSessionTTLInSeconds=1800,  # 30 minutes,
    agentResourceRoleArn=bedrock_agent_role_arn
)

agent_id = response['agent']['agentId']
print(f"Agent created. ID: {agent_id}")

Agent created. ID: MJHZPFJ31K


In [23]:
# Prepare the Agent
response = bedrock_client.prepare_agent(
    agentId=agent_id
)

print(f"Agent preparation initiated. Status: {response['agentStatus']}")

# Wait for the agent to be prepared
print("Waiting for agent to be prepared...")
prepare_status = None
while True:
    response = bedrock_client.get_agent(agentId=agent_id)
    prepare_status = response['agent']['agentStatus']
    print(f"Current status: {prepare_status}")
    
    if prepare_status == 'PREPARED':
        break
    elif prepare_status in ['FAILED', 'DELETING', 'DELETED']:
        print(f"Agent preparation failed with status: {prepare_status}")
        break
        
    time.sleep(15)

# Create an alias for the agent
response = bedrock_client.create_agent_alias(
    agentId=agent_id,
    agentAliasName="DRAFT",
    description="Alias for the product agent",
    routingConfiguration=[]
)

agent_alias_id = response['agentAlias']['agentAliasId']
print(f"Agent alias created. Alias ID: {agent_alias_id}")

Agent preparation initiated. Status: PREPARING
Waiting for agent to be prepared...
Current status: PREPARING
Current status: PREPARED
Agent alias created. Alias ID: HDPUZ46YTC


In [24]:
# Associate the knowledge base with the agent
response = bedrock_client.associate_agent_knowledge_base(
    agentId=agent_id,
    agentVersion="DRAFT",
    knowledgeBaseId=knowledge_base_id,
    description="IT Employee Handbook knowledge base"
)

print(f"Knowledge base associated with agent")

Knowledge base associated with agent


## Step 11: Create an Action Group for the Bedrock Agent for the /create function

In [25]:
# Define API schema with POST endpoint at /create - Creates OpenAPI specifications 
# for the ticket creation and lookup APIs, defining the required parameters and response formats.

api_schema = {
  "openapi": "3.0.0",
  "info": {
    "title": "ITSM Ticket Management API",
    "version": "1.0.0",
    "description": "APIs for managing ITSM tickets"
  },
  "paths": {
    "/create": {
      "post": {
        "summary": "API to create an ITSM ticket",
        "description": "Create an ITSM ticket by passing in 4 parameters in the json body: tickettype, description, impact, and urgency.",
        "operationId": "createTicket",
        "requestBody": {
          "required": "true",
          "content": {
            "application/json": {
              "schema": {
                "type": "object",
                "properties": {
                  "tickettype": {
                    "type": "string",
                    "description": "type of ticket to create, example INC0001, CHG0001, etc."
                  },
                  "description": {
                    "type": "string",
                    "description": "description of the issue or request"
                  },
                  "impact": {
                    "type": "string",
                    "description": "Impact of the issue or request"
                  },
                  "urgency": {
                    "type": "string",
                    "description": "urgency of the request"
                  }
                },
                "required": [
                  "tickettype",
                  "description",
                  "impact",
                  "urgency"
                ]
              }
            }
          }
        },
        "responses": {
          "200": {
            "description": "Ticket created successfully",
            "content": {
              "application/json": {
                "schema": {
                  "type": "object",
                  "properties": {
                    "ticketNumber": {
                      "type": "string",
                      "description": "Response from ITSM with ticket number"
                    }
                  }
                }
              }
            }
          },
          "400": {
            "description": "Bad request. One or more required fields are missing or invalid."
          }
        }
      }
    }
  }
}

In [26]:
# Create action groups: Sets up action groups that connect the agent to Lambda functions 
# for creating and looking up ITSM tickets.

response = bedrock_client.create_agent_action_group(
    agentId=agent_id,
    agentVersion="DRAFT",
    actionGroupName="TicketCreationAPI",
    actionGroupExecutor={
        'lambda': lambda_arn_create
    },
    apiSchema={
        "payload": json.dumps(api_schema)
    },
    description="Action group for ticket creation API"
)

print(f"Action group created for API")

# Step 6: Prepare the agent for use
response = bedrock_client.prepare_agent(
    agentId=agent_id
)

print(f"Agent preparation initiated")

Action group created for API
Agent preparation initiated


In [27]:
# Create an action group that allows the Agent to prompt for more information
response = bedrock_client.create_agent_action_group(
    agentId=agent_id,
    agentVersion="DRAFT",
    actionGroupName="UserInputAction",
    actionGroupState="ENABLED",
    parentActionGroupSignature="AMAZON.UserInput"
)

print(f"Action group created for API")

# Step 6: Prepare the agent for use
response = bedrock_client.prepare_agent(
    agentId=agent_id
)

print(f"Agent preparation initiated")

Action group created for API
Agent preparation initiated


## Step 12: Create an Action Group for the Bedrock Agent for the /lookup function

In [28]:
# Define API schema with GET endpoint at /lookup - Creates OpenAPI specifications for 
# the ticket creation and lookup APIs, defining the required parameters and response formats.

api_schema = {
  "openapi": "3.0.0",
  "info": {
    "title": "ITSM Ticket Management API",
    "version": "1.0.0",
    "description": "APIs for managing ITSM tickets"
  },
  "paths": {
    "/lookup": {
      "get": {
        "summary": "API to lookup the status and information for an ITSM ticket number",
        "description": "Lookup the status and information for an ITSM ticket by passing in 1 parameter in the json body: ticketNumber",
        "operationId": "lookupTicket",
        "requestBody": {
          "required": "true",
          "content": {
            "application/json": {
              "schema": {
                "type": "object",
                "properties": {
                  "ticketNumber": {
                    "type": "string",
                    "description": "the ticket number provided by the user"
                  }
                },
                "required": [
                  "ticketNumber"
                ]
              }
            }
          }
        },
        "responses": {
          "200": {
            "description": "Status and details of the ticket number lookup",
            "content": {
              "application/json": {
                "schema": {
                  "type": "object",
                  "properties": {
                    "ticketStatus": {
                      "type": "string",
                      "description": "Status of the ticket"
                    },
                    "ticketDesc": {
                      "type": "string",
                      "description": "Description or details of the ticket"
                    },
                    "ticketImpact": {
                      "type": "string",
                      "description": "Impact of the ticket"
                    },
                    "ticketUrgency": {
                      "type": "string",
                      "description": "Urgency of the ticket"
                    },  
                    "createdAt": {
                      "type": "string",
                      "description": "Date the ticket was created"
                    }                 
                  }
                }
              }
            }
          },
          "400": {
            "description": "Bad request. One or more required fields are missing or invalid."
          }
        }
      }
    }
  }
}

In [29]:
# Create an action group for the API to lookup a ticket
response = bedrock_client.create_agent_action_group(
    agentId=agent_id,
    agentVersion="DRAFT",
    actionGroupName="TicketLookupAPI",
    actionGroupExecutor={
        'lambda': lambda_arn_lookup
    },
    apiSchema={
        "payload": json.dumps(api_schema)
    },
    description="Action group for ticket lookup API"
)

print(f"Action group created for API")

# Prepare the agent for use
response = bedrock_client.prepare_agent(
    agentId=agent_id
)

print(f"Agent preparation initiated")

Action group created for API
Agent preparation initiated


### Cleanup

You have now successfully built your Amazon Bedrock Agent! To move to the next part of the lab, do the following:

- Close this notebook file.
- Return to the lab session and continue with **Task 2** to test your agent's capabilities.